# 🔬 Post-Training: IPO
Aligns the SFT model using **IPO**.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
except Exception:
    pass  # newer unsloth versions don't need this patch

# Load the SFT LoRA model directly — DO NOT call get_peft_model again!
# The model already has LoRA adapters from the SFT stage.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "../01_sft/qwen_medical_arabic_lora",
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_training(model)  # enable gradient checkpointing for training
print("Model loaded for DPO. VRAM:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")


In [ ]:
from datasets import load_dataset
import os

DATA_FILE = "../../data/alignment/05_ipo_dataset.json"
assert os.path.exists(DATA_FILE), f"Data file not found: {DATA_FILE}"

dataset = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"Dataset loaded: {len(dataset)} examples")

SYSTEM_MSG = "أنت معالج نفسي عربي خبير. ردودك آمنة ومتعاطفة وتراعي التعاليم الإسلامية السنية. لا تشخص ولا تصف أدوية."

def format_pair(example):
    prompt   = f"<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n<|im_start|>user\n{example['prompt']}<|im_end|>\n<|im_start|>assistant\n"
    chosen   = f"{example['chosen']}<|im_end|>\n"
    rejected = f"{example['rejected']}<|im_end|>\n"
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

dataset = dataset.map(format_pair, remove_columns=dataset.column_names)
print(f"Formatted. Sample count: {len(dataset)}")
print("Sample prompt (first 150 chars):")
print(dataset[0]["prompt"][:150])


In [ ]:
from trl import DPOTrainer, DPOConfig

training_args = DPOConfig(
    output_dir          = "outputs_ipo",
    beta                = 0.1,
    loss_type           = "ipo",
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    learning_rate       = 5e-5,
    lr_scheduler_type   = "cosine",
    warmup_ratio        = 0.1,
    max_length          = 1024,
    max_prompt_length   = 512,
    optim               = "adamw_8bit",
    fp16                = not is_bfloat16_supported(),
    bf16                = is_bfloat16_supported(),
    num_train_epochs    = 2,
    logging_steps       = 10,
    save_steps          = 50,
    save_total_limit    = 2,
    report_to           = "none",
)

trainer = DPOTrainer(
    model        = model,
    ref_model    = None,
    args         = training_args,
    train_dataset= dataset,
    tokenizer    = tokenizer,
)
print("IPO Trainer ready!")


In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("qwen_medical_arabic_ipo")
tokenizer.save_pretrained("qwen_medical_arabic_ipo")
print("Saved IPO model -> qwen_medical_arabic_ipo/")
